In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS iics.governance_poc;

In [0]:
%sql
SELECT COUNT(*) as cnt FROM system.access.audit

In [0]:
%sql
DESCRIBE TABLE system.access.audit

In [0]:
spark.sql("DESCRIBE TABLE system.access.audit").display()

In [0]:
%sql
INSERT INTO governance.user_activity_log
SELECT
  event_time,
  event_date,
  workspace_id,
  user_identity.email AS user_email,
  'audit' AS action_source,
  action_name AS action_type,
  service_name,
  request_params['catalog_name'] AS catalog_name,
  request_params['schema_name'] AS schema_name,
  request_params['table_name'] AS table_name,
  NULL AS statement_id,
  NULL AS statement_text,
  NULL AS execution_status,
  sourceipaddress AS source_ip,
  to_json(request_params) AS details
FROM system.access.audit

In [0]:
%sql

CREATE TABLE IF NOT EXISTS governance.user_activity_log (
  event_time TIMESTAMP,
  event_date DATE,
  workspace_id STRING,
  user_email STRING,
  action_source STRING,       -- audit, query_history, table_lineage
  action_type STRING,         -- SELECT, INSERT, UPDATE, DELETE, ALTER, DROP...
  service_name STRING,
  catalog_name STRING,
  schema_name STRING,
  table_name STRING,
  statement_id STRING,
  statement_text STRING,
  execution_status STRING,
  source_ip STRING,
  details STRING
)
USING DELTA;

In [0]:
%sql
DESCRIBE HISTORY iics.casemix.healthcare_ingestion_test;

In [0]:
df = spark.sql("""SELECT
'iics' as catalog,
'casemix' as schema,
'healthcare_ingestion_test' as table,
*
FROM (
  DESCRIBE HISTORY iics.casemix.healthcare_ingestion_test
)
ORDER BY version DESC""")

In [0]:
df.write.mode("overwrite").saveAsTable("iics.governance_poc.audit_logs_test")

In [0]:
%sql
SELECT * FROM iics.governance_poc.audit_logs_test

In [0]:
%sql
SELECT
  version,
  timestamp,
  userName AS executed_by,
  operation,
  operationParameters,
  operationMetrics,
  job,
  notebook,
  clusterId,
  readVersion,
  userMetadata,
  engineInfo
FROM (
  DESCRIBE HISTORY iics.casemix.healthcare_ingestion_test
)
ORDER BY version DESC;

In [0]:
%sql
SELECT
  l.event_time,
  l.created_by,
  q.statement_type,
  q.execution_status,
  q.statement_text,
  l.source_table_full_name,
  l.target_table_full_name
FROM system.access.table_lineage l
LEFT JOIN system.query.history q
  ON l.statement_id = q.statement_id
WHERE l.source_table_full_name =
        'iics.casemix.healthcare_ingestion_test'
   OR l.target_table_full_name =
        'iics.casemix.healthcare_ingestion_test'
ORDER BY l.event_time DESC;

In [0]:
%sql
SELECT * FROm iics.casemix.healthcare_ingestion_test limit 10;

In [0]:
%sql
ALTER TABLE iics.casemix.healthcare_ingestion_test
SET TBLPROPERTIES ('delta.enableChangeDataFeed' = true);

In [0]:
%sql
DESCRIBE HISTORY iics.casemix.healthcare_ingestion_test;

In [0]:
%sql
SELECT * FROM iics.casemix.healthcare_ingestion_test where CASE_SURR_ID='6'

In [0]:
%sql
delete from iics.casemix.healthcare_ingestion_test where CASE_SURR_ID='6'

In [0]:
%sql
SELECT * FROM iics.casemix.healthcare_ingestion_test where CASE_SURR_ID='8000001'

In [0]:
%sql
SELECT 
case_surr_id,
_change_type,
_commit_version,
_commit_timestamp
FROM table_changes(
  'iics.casemix.healthcare_ingestion_test',
  0
)
WHERE 
_change_type = 'delete'
ORDER BY _commit_timestamp DESC;

In [0]:
df = spark.read.format("csv").option("inferSchema","true").option("header","true").option("delimiter","|").load("/Volumes/iics/casemix/history/csv_files_test/case_data_001.csv")

In [0]:
display(df)

In [0]:
df.write \
  .mode("overwrite") \
  .format("delta") \
  .option("delta.enableChangeDataFeed", "true") \
  .saveAsTable("iics.casemix.healthcare_ingestion_test")

In [0]:
%sql
describe history iics.casemix.healthcare_ingestion_test

In [0]:
%sql
INSERT INTO iics.casemix.patient_details_test (customer_id, name, dob, status)
VALUES 
(1004, 'WILLIAM', to_date('20-JAN-1990', 'dd-MMM-yyyy'), 'ACTIVE')
;

In [0]:
%sql
DELETE FROM iics.casemix.patient_details_test
WHERE customer_id = 1004;

In [0]:
%sql
SELECT * FROM iics.casemix.audit_test

In [0]:
%sql
select * from iics.casemix.audit_test

In [0]:
%sql
DELETE FROM iics.casemix.patient_details_test
WHERE customer_id = 1001;